# RTL Combinational Depth Predictor
## Exploratory Data Analysis

**Girl Hackathon 2025 — Silicon Track**

This notebook covers:
1. Dataset overview
2. Feature distributions
3. Correlation analysis
4. Model comparison
5. Interactive prediction

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib, json
from pathlib import Path

plt.rcParams.update({'figure.facecolor': '#1a1a2e', 'axes.facecolor': '#16213e',
                     'text.color': '#eaeaea', 'axes.labelcolor': '#eaeaea',
                     'xtick.color': '#eaeaea', 'ytick.color': '#eaeaea',
                     'grid.color': '#0f3460', 'axes.edgecolor': '#0f3460',
                     'font.family': 'sans-serif'})

PROJECT_ROOT = Path('..').resolve()
DATASET_PATH = PROJECT_ROOT / 'data' / 'dataset.csv'
print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset: {DATASET_PATH}')

## 1. Dataset Overview

In [ ]:
df = pd.read_csv(DATASET_PATH)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(10)

In [ ]:
df.describe()

## 2. Depth Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['actual_depth'], bins=20, color='#e94560', edgecolor='#0f3460', alpha=0.9)
axes[0].set_title('Depth Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Combinational Depth (gate levels)')
axes[0].set_ylabel('Count')
axes[0].grid(True, alpha=0.4)

op_counts = df['op_type'].value_counts()
colors = ['#e94560', '#0f3460', '#06d6a0', '#ffd166', '#ef476f', '#118ab2', '#073b4c']
axes[1].bar(op_counts.index, op_counts.values, color=colors[:len(op_counts)], alpha=0.9)
axes[1].set_title('Operator Type Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Operator')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('../results/evaluation_plots/depth_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Correlation Heatmap

In [ ]:
numeric_cols = ['fanin','fanout','signal_width','op_complexity','nesting_depth',
                'operation_count','has_mul','has_add','in_loop','is_registered',
                'module_input_count','conditional_mux_count','actual_depth']

corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(220, 20, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, annot=True, fmt='.2f',
            annot_kws={'size': 8}, ax=ax, vmax=1, vmin=-1,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/evaluation_plots/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Depth by Operator Type

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

op_groups = df.groupby('op_type')['actual_depth']
ops_sorted = op_groups.mean().sort_values(ascending=False)

for i, (op, mean_depth) in enumerate(ops_sorted.items()):
    depths = df[df['op_type'] == op]['actual_depth']
    ax.scatter([i] * len(depths), depths, alpha=0.5, s=40, color='#e94560')
    ax.plot(i, mean_depth, marker='D', markersize=10, color='#06d6a0')

ax.set_xticks(range(len(ops_sorted)))
ax.set_xticklabels(ops_sorted.index, rotation=45)
ax.set_title('Depth by Operator Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Operator')
ax.set_ylabel('Combinational Depth')
ax.grid(True, axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('../results/evaluation_plots/depth_by_operator.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Model Comparison

In [ ]:
results_path = PROJECT_ROOT / 'results' / 'model_comparison.json'
if results_path.exists():
    results = json.loads(results_path.read_text())
    df_res = pd.DataFrame(results).T.reset_index().rename(columns={'index': 'Model'})

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    for ax, col, title in zip(axes,
        ['MAE', 'RMSE', 'Accuracy_within_1'],
        ['Mean Absolute Error', 'RMSE', 'Accuracy within +/-1']):

        bars = ax.bar(df_res['Model'], df_res[col].astype(float),
                      color='#e94560', edgecolor='#0f3460', alpha=0.9)
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.tick_params(axis='x', rotation=45)
        ax.grid(True, axis='y', alpha=0.4)
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.savefig('../results/evaluation_plots/model_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    display(df_res[['Model','MAE','RMSE','Accuracy_within_1','Accuracy_within_2','CV_MAE_5fold']])
else:
    print('Run python src/train_models.py first')

## 6. Interactive Prediction

In [ ]:
from feature_extraction import VerilogFeatureExtractor
from predict import predict_depth

# ---- Change these values ----
VERILOG_FILE = '../data/rtl_designs/local/array_multiplier_8bit.v'
SIGNAL_NAME  = 'product'
CLOCK_NS     = 2.0   # 500 MHz
# -----------------------------

result = predict_depth(VERILOG_FILE, SIGNAL_NAME, clock_period_ns=CLOCK_NS)

print(f'Signal       : {result["signal"]}')
print(f'Module       : {result["module"]}')
print(f'Depth        : {result["predicted_depth"]} gate levels')
print(f'Est. Delay   : {result["estimated_delay_ns"]} ns')
print(f'Timing OK    : {result["timing_ok"]}')
print(f'Inference    : {result["inference_ms"]} ms')
print()
print('Features:', result['features'])